# A1.2: So sánh CNN và Transformer trên Food-101

Notebook này thực hiện phần A1.2 theo đúng protocol nhóm đã đặt ra ở A1.1. Protocol đặt ba câu hỏi:

1. Pretrain giúp CNN và ViT đến mức nào trên một tập khoảng 75 nghìn ảnh train?
2. Mức freeze nào cân bằng tốt nhất giữa accuracy và chi phí (thời gian, tham số trainable, bộ nhớ)?
3. (Mở rộng) Transformer phân cấp như Swin có bớt phụ thuộc vào pretrain hơn ViT thuần không?

Cặp chính là **ResNet-50** (CNN, 25,6M tham số) và **DeiT-Small** (ViT thuần, 22,1M tham số). Cả hai dùng trọng số ImageNet-1k từ `timm` và có chi phí tính toán gần nhau (khoảng 4 GFLOPs mỗi ảnh 224×224), nên chênh lệch kết quả chủ yếu đến từ kiến trúc chứ không phải do một bên lớn hơn.

Về lý thuyết, CNN có sẵn hai thiên kiến quy nạp (inductive bias) mạnh: tính cục bộ (mỗi kernel chỉ nhìn một vùng nhỏ) và tính bất biến tịnh tiến (cùng một kernel trượt trên toàn ảnh). Nhờ vậy CNN học được khá tốt từ lượng dữ liệu vừa phải. ViT chia ảnh thành các patch 16×16 và để self-attention nhìn toàn ảnh ngay từ lớp đầu. Cách này linh hoạt hơn nhưng mô hình phải tự học các quy luật không gian từ dữ liệu, nên thường cần nhiều dữ liệu hoặc augmentation và regularization mạnh; DeiT ra đời chính để giảm bớt nhu cầu này. Vì thế cặp ResNet-50 và DeiT-S cho thấy rõ nhất câu hỏi "khi nào ViT cần pretrain": nhóm kỳ vọng khoảng cách giữa pretrained và from-scratch của DeiT-S lớn hơn của ResNet-50.

Food-101 gồm 101 món ăn, mỗi lớp 750 ảnh train và 250 ảnh test. Ảnh test đã được kiểm tra thủ công, còn ảnh train được giữ nguyên nhiễu (một số nhãn sai, màu sắc lệch), nên đây là bộ dữ liệu khá sát thực tế. Nhiều lớp trông rất giống nhau (steak và filet mignon, chocolate cake và chocolate mousse), thuận lợi để phân tích mỗi kiến trúc nhầm ở đâu.

Mỗi mô hình được thử ở bốn chế độ, đi từ "không dùng gì từ ImageNet" đến "điều chỉnh toàn bộ mạng":

| ID | Mô hình | Chế độ | Epoch | Seed | Chế độ này đo điều gì |
|---|---|---|---|---|---|
| R1 / R5 | ResNet-50 / DeiT-S | From-scratch | 100 | 42 | Khả năng học thuần từ Food-101, không có pretrained |
| R2 / R6 | ResNet-50 / DeiT-S | Freeze toàn bộ | 30 | 42, 43, 44 | Chất lượng feature ImageNet khi chỉ train lớp phân loại (linear probe) |
| R3 / R7 | ResNet-50 / DeiT-S | Freeze một phần | 15 | 42, 43, 44 | Mở khóa nửa sau mạng (layer4 / block 6–11) cho thích nghi với món ăn |
| R4 / R8 | ResNet-50 / DeiT-S | Full finetune | 15 | 42, 43, 44 | Điều chỉnh toàn bộ mạng |

Protocol chi tiết: `reports/A1.1_Dataset-Protocol.pdf`. Theo protocol, mỗi dòng trong bảng là một "run" (8 run bắt buộc). Tính cả các seed lặp lại thì có 20 lần huấn luyện.

**Trạng thái.** Toàn bộ phần bắt buộc đã chạy xong: 20 lần huấn luyện cho bảng chính và 4 lần dò learning rate, tổng cộng khoảng 8,4 giờ A100. Kết quả nằm trong `runs/`, hình trong `figures/`, phân tích đầy đủ trong `reports/A1.2_Report.tex`.

## 1. Kết nối Drive và cấu hình

Code và kết quả được đặt trên Google Drive để không bị mất khi Colab ngắt phiên. Riêng ảnh thì không đọc trực tiếp từ Drive: Drive được mount qua mạng, đọc hơn 100 nghìn file nhỏ từ đó sẽ khiến GPU phải chờ dữ liệu. Vì vậy mỗi phiên, ảnh được copy về ổ cục bộ `/content/data`.

Do team dùng Google Colab để train nên có đặt ra biến `HOURS` dùng khi muốn giới hạn thời gian một phiên: vòng train sẽ dừng ở cuối một epoch, sau khi đã lưu checkpoint, thay vì bị cắt ngang giữa chừng.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Thac si/Hoc ki 3/Hoc sau va ung dung/Exercises/SDH_HK261_CO5085/A1')
DATA_ROOT = Path('/content/data')                                   # ổ local Colab, không đọc ảnh từ Drive
DRIVE_TAR = Path('/content/drive/MyDrive/datasets/food-101.tar.gz')  # bản lưu archive để phiên sau copy nhanh
HOURS = None          # None = chạy hết hàng đợi; đặt số giờ để dừng ở ranh giới epoch
NUM_WORKERS = os.cpu_count()

assert (ROOT / 'src' / 'train.py').is_file(), ROOT
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
OUT, SPLITS = Path('runs'), Path('splits')
print('A1:', ROOT, '| workers:', NUM_WORKERS)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
A1: /content/drive/MyDrive/Thac si/Hoc ki 3/Hoc sau va ung dung/Exercises/SDH_HK261_CO5085/A1 | workers: 12


## 2. Môi trường

Colab đã cài sẵn PyTorch và torchvision, nên nhóm chỉ bổ sung `timm` (thư viện mô hình) và vài gói phụ nếu thiếu. Nhóm không cài lại torch để tránh lệch phiên bản với CUDA của máy.

Các run chính thức chạy trên A100 vì GPU này hỗ trợ bf16. Huấn luyện bằng mixed precision bf16 nhanh hơn fp32 đáng kể, và khác với fp16 là không cần loss scaling vì bf16 có cùng dải giá trị với fp32. Trên GPU khác code vẫn chạy (tự chuyển sang fp16), nhưng thời gian train và throughput sẽ không so sánh được với các run đã có.

In [ ]:
import importlib.util
missing = [p for p, m in [('timm', 'timm'), ('pyyaml', 'yaml'), ('tabulate', 'tabulate'), ('scikit-learn', 'sklearn')]
           if importlib.util.find_spec(m) is None]
if missing:
    %pip install -q {' '.join(missing)}

import torch, timm
assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU (A100)'
gpu = torch.cuda.get_device_name(0)
print('torch', torch.__version__, '| timm', timm.__version__, '|', gpu,
      f'| {torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GiB')
if 'A100' not in gpu:
    print('CẢNH BÁO: protocol dùng A100 cho run chính thức; GPU này chỉ nên để debug.')

torch 2.14.1+cu130 | timm 1.0.31 | NVIDIA A100-SXM4-40GB | 39 GiB


## 3. Dữ liệu Food-101 và tập validation

Lần đầu, cell tải archive (khoảng 5 GB) từ ETH Zürich và lưu một bản lên Drive; các phiên sau copy lại từ Drive, nhanh hơn và không phụ thuộc máy chủ ETH. Mã md5 được kiểm tra trước khi giải nén để không vô tình dùng file tải dở.

Food-101 chỉ có hai tập train và test, nên nhóm tách 10% ảnh train làm validation theo kiểu phân tầng (mỗi lớp giữ đúng tỉ lệ), với seed 42. Split được lưu vào `splits/` và dùng chung cho mọi run. Validation dùng để chọn epoch tốt nhất và so sánh các cấu hình; tập test chỉ được đánh giá một lần trên checkpoint đã chọn. Làm vậy để không vô tình "tinh chỉnh theo test" rồi báo cáo một con số lạc quan hơn thực tế.

In [ ]:
import hashlib, shutil, tarfile, time
from src.data import load_food101, make_splits

FOOD_MD5 = '85eeb15f3717b99a5da872d97d918f87'
t0 = time.time()
DATA_ROOT.mkdir(parents=True, exist_ok=True)
local_tar = DATA_ROOT / 'food-101.tar.gz'
if not (DATA_ROOT / 'food-101' / 'images').is_dir():
    if DRIVE_TAR.is_file() and not local_tar.is_file():
        print('Copy archive từ Drive...'); shutil.copy(DRIVE_TAR, local_tar)
    if local_tar.is_file():
        md5 = hashlib.md5()
        with open(local_tar, 'rb') as f:
            for chunk in iter(lambda: f.read(1 << 24), b''):
                md5.update(chunk)
        assert md5.hexdigest() == FOOD_MD5, 'md5 sai: archive hỏng, xóa và tải lại'
        print('Giải nén...')
        with tarfile.open(local_tar) as tar:
            tar.extractall(DATA_ROOT, **({'filter': 'data'} if hasattr(tarfile, 'data_filter') else {}))
    else:
        load_food101(str(DATA_ROOT), 'train', download=True)   # tải + giải nén
        DRIVE_TAR.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(local_tar, DRIVE_TAR)
n_img = sum(1 for _ in (DATA_ROOT / 'food-101' / 'images').rglob('*.jpg'))
assert n_img == 101000, f'Chỉ thấy {n_img}/101000 ảnh'
if not (SPLITS / 'train_idx.json').is_file():
    make_splits(str(DATA_ROOT), str(SPLITS), val_frac=0.1, seed=42)
print(f'Sẵn sàng sau {time.time() - t0:.0f}s: {n_img} ảnh, split tại {SPLITS}/')

Sẵn sàng sau 0s: 101000 ảnh, split tại splits/


## 4. Khám phá dữ liệu

Hai hình dưới đây là phân bố kích thước ảnh và một ảnh mẫu của 24 lớp ngẫu nhiên. Ảnh Food-101 có kích thước không đồng đều (cạnh dài nhất đã được thu về 512 px), nên mọi mô hình đều đưa ảnh về 224×224: lúc train dùng RandomResizedCrop, lúc đánh giá dùng Resize 256 rồi CenterCrop 224 theo quy ước của ImageNet. Lưới ảnh mẫu cũng cho thấy nhiều món có màu sắc và cách bày gần nhau, gợi ý trước những cặp lớp dễ bị nhầm.

In [ ]:
import json, random
import numpy as np, matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown, Image as IImage

FIG = Path('figures'); FIG.mkdir(exist_ok=True)
if (FIG / 'eda_sizes.png').exists() and (FIG / 'eda_samples.png').exists():
    for name in ['eda_sizes', 'eda_samples']:
        display(IImage(str(FIG / f'{name}.png')))
else:
    base = load_food101(str(DATA_ROOT), 'train')
    classes = base.classes
    labels = np.array(base._labels)
    print('Số lớp:', len(classes), '| ảnh train chính thức:', len(base),
          '| ảnh/lớp:', np.bincount(labels).min(), '-', np.bincount(labels).max())
    random.seed(0)
    sizes = np.array([Image.open(base._image_files[i]).size for i in random.sample(range(len(base)), 2000)])
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].hist(sizes[:, 0], bins=40); ax[0].set_title('Chiều rộng (px)')
    ax[1].hist(sizes[:, 1], bins=40); ax[1].set_title('Chiều cao (px)')
    fig.tight_layout(); fig.savefig(FIG / 'eda_sizes.png', dpi=130); plt.show()
    pick = random.sample(range(len(classes)), 24)
    fig, axes = plt.subplots(4, 6, figsize=(15, 10))
    for ax, c in zip(axes.flat, pick):
        i = int(np.where(labels == c)[0][0])
        ax.imshow(Image.open(base._image_files[i]).convert('RGB')); ax.set_title(classes[c], fontsize=8); ax.axis('off')
    fig.tight_layout(); fig.savefig(FIG / 'eda_samples.png', dpi=110); plt.show()

Output hidden; open in https://colab.research.google.com to view.

## 5. Kế hoạch chạy và dò learning rate

Tám run của cặp chính, tính cả seed lặp là 20 lần huấn luyện: 6 run pretrained mỗi run 3 seed, 2 run from-scratch mỗi run 1 seed. Thứ tự ưu tiên là chạy đủ 8 run với seed 42 trước để có bức tranh so sánh hoàn chỉnh, rồi mới thêm seed 43 và 44 để ước lượng độ dao động. From-scratch chỉ chạy một seed vì mỗi run 100 epoch đã mất khoảng hai giờ trên A100.

Siêu tham số theo bảng ở mục 7 của protocol: AdamW, warmup rồi giảm learning rate theo cosine, batch 256, bf16. Khi fine-tune, lớp phân loại mới dùng learning rate 1e-3, lớn hơn backbone: phần pretrained chỉ cần điều chỉnh nhẹ, còn lớp đầu ra khởi tạo ngẫu nhiên cần học nhanh hơn. From-scratch dùng augmentation mạnh (RandAugment, RandomErasing, Mixup/CutMix, label smoothing) theo công thức của DeiT, vì ViT thuần không có inductive bias không gian nên rất dễ overfit trên 68 nghìn ảnh; ResNet-50 dùng cùng công thức để khác biệt chỉ đến từ kiến trúc.

**Dò learning rate backbone.** Protocol yêu cầu không cố định LR backbone cho fine-tune mà thử ba giá trị {5e-5, 1e-4, 3e-4} với full finetune seed 42, chọn giá trị có validation top-1 cao nhất cho từng mô hình, rồi dùng giá trị đó cho freeze một phần. Lý do là hai kiến trúc có thể hợp với các LR khác nhau: nếu dùng chung một LR thì kết quả vô tình nghiêng về mô hình hợp với LR đó hơn. Hai mô hình được dò với cùng ngân sách nên vẫn công bằng. Nhóm dùng luôn LR đã chọn cho seed 43/44 của full finetune, để mean ± std được đo ở đúng một cấu hình. Điểm 1e-4 chính là `R4_s42` và `R8_s42` đã chạy, nên chỉ cần thêm 4 run, khoảng 1,2 giờ.

Khi dò xong, hai mô hình chọn hai giá trị khác nhau:

| Mô hình | 5e-5 | 1e-4 | 3e-4 | Chọn |
|---|---|---|---|---|
| ResNet-50 | 75,12 | 77,91 | **81,35** | 3e-4 |
| DeiT-S | 83,51 | **84,63** | 84,58 | 1e-4 |

(validation top-1, %, full finetune seed 42)

ResNet-50 tăng đều theo LR, còn DeiT-S gần như phẳng từ 1e-4 trở lên. Nếu giữ chung 1e-4 như bản protocol ban đầu, ResNet-50 sẽ bị đánh giá thấp khoảng 3,5 điểm top-1, và khoảng cách giữa hai kiến trúc ở full finetune bị phóng đại hơn gấp đôi. Đây là lý do bước dò LR được đưa vào protocol.

Lịch chạy (`src/plan.py`) tự xử lý phần phụ thuộc này: R3/R7 và seed lặp của R4/R8 chỉ được mở khi đủ ba điểm dò cho mỗi mô hình. Vì LR của ResNet-50 khác 1e-4, các run R3/R4 dùng thư mục có hậu tố `_lr0.0003`, và bảng tổng hợp chỉ lấy run ở LR đã chọn. Run `R3_s42` chạy dở với 1e-4 (dừng ở epoch 4) được bỏ.

Cell dưới in trạng thái từng run và kết quả dò LR đọc từ `runs/`. Muốn chạy riêng vài run thì đặt `ONLY`.

In [ ]:
import importlib, src.plan, src.train
importlib.reload(src.plan); importlib.reload(src.train)
from src.plan import plan, next_job, sweep_table, selected_lr

def show_plan():
    for row in plan(OUT):
        state = row['state']
        if state == 'dở':
            done = sum(1 for _ in open(OUT / row['dir'] / 'history.csv')) - 1
            state = f'dở, sẽ chạy tiếp từ epoch {done + 1}'
        lr = '' if row['lr'] is None else f"lr_backbone={row['lr']:g}"
        print(f"{row['dir']:<18} {lr:<20} {state}")

show_plan()
print('\nDò LR (val top-1):', {m: {f'{lr:g}': v for lr, v in row.items()} for m, row in sweep_table(OUT).items()})
print('LR đã chọn:', selected_lr(OUT) or 'chưa đủ 3 điểm cho mỗi mô hình')

# Mặc định chạy theo thứ tự trong plan. Muốn chạy riêng vài thư mục thì đặt ví dụ ONLY = {'R1_s42', 'R5_s42'}.
ONLY = None

R2_s42                                  xong
R6_s42                                  xong
R4_s42_lr5e-05     lr_backbone=5e-05    chưa chạy
R4_s42                                  xong
R4_s42_lr0.0003    lr_backbone=0.0003   chưa chạy
R8_s42_lr5e-05     lr_backbone=5e-05    chưa chạy
R8_s42                                  xong
R8_s42_lr0.0003    lr_backbone=0.0003   chưa chạy
R3_s42                                  chờ dò LR
R7_s42                                  chờ dò LR
R1_s42                                  chưa chạy
R5_s42                                  chưa chạy
R2_s43                                  xong
R6_s43                                  xong
R3_s43                                  chờ dò LR
R7_s43                                  chờ dò LR
R4_s43                                  chờ dò LR
R8_s43                                  chờ dò LR
R2_s44                                  xong
R6_s44                                  xong
R3_s44                                  

## 6. Huấn luyện

Mỗi run được hàm `run_one` trong `src/train.py` xử lý trọn vẹn: dựng mô hình, train, chọn checkpoint theo validation top-1, đánh giá test một lần rồi ghi `metrics.json`. Trên A100, mỗi run fine-tune 15 epoch mất khoảng 17 phút, mỗi run from-scratch 100 epoch khoảng 1 giờ 50 phút.

Có hai chi tiết thiết kế đáng nhắc:

- **Freeze toàn bộ:** backbone không thay đổi trong suốt quá trình train, nên nhóm trích feature của toàn bộ ảnh một lần rồi chỉ train lớp tuyến tính trên feature đó. Mỗi epoch vì thế chỉ mất chưa tới một giây thay vì hơn một phút. Đổi lại, ảnh train ở chế độ này không được augmentation ngẫu nhiên.
- **BatchNorm khi đóng băng:** các lớp BatchNorm thuộc phần bị đóng băng được giữ ở chế độ eval, để mean/variance học từ ImageNet không bị ghi đè bởi thống kê của từng batch Food-101. Điều này chỉ ảnh hưởng ResNet; DeiT dùng LayerNorm nên không có vấn đề này.

Checkpoint `last.pt` được lưu sau mỗi epoch, ghi ra file tạm rồi mới đổi tên nên dù bị ngắt giữa chừng file cũ vẫn nguyên vẹn. Nếu Colab ngắt kết nối, chỉ cần chạy lại các cell 1, 2, 3, 5, 6: run đã xong được bỏ qua, run dở tiếp tục từ epoch đã lưu.

In [ ]:
import gc, time
from types import SimpleNamespace
from src.train import run_one, BudgetExhausted

def make_args(**override):
    a = dict(data_root=str(DATA_ROOT), out_root=str(OUT), splits_dir=str(SPLITS), num_workers=NUM_WORKERS,
             epochs=None, lr_backbone=None, batch_size=None, smoke=False, force=False,
             deadline=time.monotonic() + HOURS * 3600 if HOURS else None)
    a.update(override)
    return SimpleNamespace(**a)

deadline = time.monotonic() + HOURS * 3600 if HOURS else None
skip, t_start = set(), time.time()
try:
    # Tính lại job kế tiếp sau mỗi run: khi dò LR xong, R3/R7 và seed lặp của R4/R8 mới được mở.
    while (job := next_job(OUT, skip)) is not None:
        if ONLY is not None and job['dir'] not in ONLY:
            skip.add(job['dir']); continue
        t0 = time.time()
        try:
            run_one(job['run_id'], job['seed'], make_args(lr_backbone=job['lr'], deadline=deadline))
        finally:
            gc.collect(); torch.cuda.empty_cache()
        if (OUT / job['dir'] / 'metrics.json').is_file():
            print(f"--- {job['dir']}: {(time.time() - t0) / 60:.1f} phút ---")
        else:
            skip.add(job['dir'])
except BudgetExhausted as e:
    print('Hết số giờ đặt trong HOURS:', e)
except KeyboardInterrupt:
    print('Đã dừng tay. Chạy lại cell để chạy tiếp.')
print(f'Phiên này: {(time.time() - t_start) / 3600:.2f} giờ\n')
show_plan()

=== R4 seed 42: resnet50 / full | NVIDIA A100-SXM4-40GB | amp=torch.bfloat16 ===


model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

[R4 s42] ep 1/15 train_loss=4.3655 val_top1=0.3608 (73s train, 81s tổng)
[R4 s42] ep 2/15 train_loss=2.8290 val_top1=0.5714 (60s train, 66s tổng)
[R4 s42] ep 3/15 train_loss=2.0851 val_top1=0.6409 (62s train, 69s tổng)
[R4 s42] ep 4/15 train_loss=1.8205 val_top1=0.6776 (61s train, 68s tổng)
[R4 s42] ep 5/15 train_loss=1.6855 val_top1=0.7002 (61s train, 68s tổng)
[R4 s42] ep 6/15 train_loss=1.5843 val_top1=0.7133 (61s train, 68s tổng)
[R4 s42] ep 7/15 train_loss=1.5215 val_top1=0.7277 (61s train, 68s tổng)
[R4 s42] ep 8/15 train_loss=1.4679 val_top1=0.7332 (61s train, 68s tổng)
[R4 s42] ep 9/15 train_loss=1.4261 val_top1=0.7414 (61s train, 68s tổng)
[R4 s42] ep 10/15 train_loss=1.4030 val_top1=0.7442 (61s train, 67s tổng)
[R4 s42] ep 11/15 train_loss=1.3790 val_top1=0.7465 (61s train, 69s tổng)
[R4 s42] ep 12/15 train_loss=1.3647 val_top1=0.7498 (61s train, 68s tổng)
[R4 s42] ep 13/15 train_loss=1.3477 val_top1=0.7485 (61s train, 68s tổng)
[R4 s42] ep 14/15 train_loss=1.3484 val_top1=0.

model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

[R8 s42] ep 1/15 train_loss=3.4472 val_top1=0.6071 (60s train, 67s tổng)
[R8 s42] ep 2/15 train_loss=1.7435 val_top1=0.7302 (59s train, 66s tổng)
[R8 s42] ep 3/15 train_loss=1.3985 val_top1=0.7666 (60s train, 66s tổng)
[R8 s42] ep 4/15 train_loss=1.2319 val_top1=0.7864 (60s train, 67s tổng)
[R8 s42] ep 5/15 train_loss=1.1336 val_top1=0.8015 (60s train, 67s tổng)
[R8 s42] ep 6/15 train_loss=1.0553 val_top1=0.8067 (60s train, 67s tổng)
[R8 s42] ep 7/15 train_loss=0.9926 val_top1=0.8149 (59s train, 67s tổng)
[R8 s42] ep 8/15 train_loss=0.9474 val_top1=0.8214 (60s train, 66s tổng)
[R8 s42] ep 9/15 train_loss=0.8993 val_top1=0.8240 (60s train, 67s tổng)
[R8 s42] ep 10/15 train_loss=0.8641 val_top1=0.8243 (59s train, 66s tổng)
[R8 s42] ep 11/15 train_loss=0.8290 val_top1=0.8294 (60s train, 67s tổng)
[R8 s42] ep 12/15 train_loss=0.8171 val_top1=0.8319 (60s train, 67s tổng)
[R8 s42] ep 13/15 train_loss=0.7989 val_top1=0.8342 (60s train, 66s tổng)
[R8 s42] ep 14/15 train_loss=0.7903 val_top1=0.

## 7. (Tuỳ chọn) Cặp mô hình mở rộng X1–X8

Cặp ConvNeXt-T và Swin-T trả lời câu hỏi thứ ba của protocol. Hai kiến trúc này tiến lại gần nhau từ hai phía: ConvNeXt là CNN được thiết kế lại theo hướng của Swin, còn Swin là Transformer đưa tính cục bộ trở lại bằng attention trong từng cửa sổ và cấu trúc phân cấp. So sánh hai cặp cho biết khác biệt đến từ "convolution hay attention" hay từ "có hay không có inductive bias cục bộ". Chỉ chạy phần này khi phần bắt buộc đã xong.

In [ ]:
RUN_EXTENDED = False
if RUN_EXTENDED:
    for run in ['X2', 'X6', 'X4', 'X8', 'X3', 'X7', 'X1', 'X5']:
        run_one(run, 42, make_args()); gc.collect(); torch.cuda.empty_cache()

## 8. Tổng hợp kết quả và hình cho báo cáo

`src/summarize.py` gom `metrics.json` của mọi run (bỏ qua smoke test; với R3/R4/R7/R8 chỉ lấy run ở LR đã chọn) thành bảng trung bình ± độ lệch chuẩn qua các seed, xuất ra cả Markdown (`summary.md`) và LaTeX (`summary.tex`). Các hình bám theo kế hoạch phân tích ở mục 8 của protocol:

- `lr_sweep`: validation top-1 theo LR backbone của full finetune, cho thấy LR nào được chọn cho từng mô hình.
- `mode_compare`: ở từng chế độ, CNN hay ViT tốt hơn, và chênh lệch đó lớn hơn hay nhỏ hơn độ dao động giữa các seed.
- `curves_by_mode`: đường loss và accuracy của train và validation theo epoch, để xem mô hình hội tụ thế nào và có overfit không.
- `pretrain_gap`: pretrained giúp mỗi kiến trúc được bao nhiêu so với train từ đầu. Đây là câu hỏi thứ nhất của protocol.
- `tradeoff`, `efficiency`: đánh đổi giữa độ chính xác và chi phí (số tham số cần train, thời gian, VRAM, tốc độ suy luận). Đây là câu hỏi thứ hai.
- `confusion_*` và bảng 10 cặp lớp hay nhầm nhất; `misclassified_*` là lưới ảnh đoán sai, dùng để tự phân loại nguyên nhân (món giống nhau, ảnh có nhiều món, nhãn sai) trong báo cáo.
- `agreement` kèm kiểm định McNemar, và `per_class`: phân tích thêm ngoài protocol. Vì hai mô hình được đánh giá trên cùng một tập test, McNemar cho biết chênh lệch có ý nghĩa thống kê hay không, chặt chẽ hơn việc chỉ so hai con số accuracy.

Khi cần chọn "run tốt nhất" của một mô hình, nhóm luôn chọn theo validation để tập test không tham gia vào việc lựa chọn.

Script được gọi bằng `runpy` ngay trong kernel (không dùng `%run -m`, vì magic này không chuyển các tham số `--out_root`... cho script). Lưới ảnh đoán sai cần ảnh test nên phải chạy cell 3 trước.

In [13]:
import runpy
sys.argv = ['summarize', '--out_root', 'runs', '--splits_dir', 'splits', '--fig_dir', 'figures', '--data_root', str(DATA_ROOT)]
runpy.run_module('src.summarize', run_name='__main__')
display(Markdown(Path('summary.md').read_text(encoding='utf-8')))
for name in ['lr_sweep', 'mode_compare', 'curves_by_mode', 'pretrain_gap', 'tradeoff', 'efficiency', 'agreement', 'per_class',
             'confusion_resnet50', 'confusion_deit_s', 'misclassified_resnet50', 'misclassified_deit_s']:
    p = FIG / f'{name}.png'
    if p.exists():
        print(name); display(IImage(str(p)))

Output hidden; open in https://colab.research.google.com to view.

### Kết quả chính

| Chế độ | ResNet-50 | DeiT-S |
|---|---|---|
| From-scratch (1 seed) | **82,48** | 67,13 |
| Freeze toàn bộ | 71,06 ± 0,05 | **72,79 ± 0,09** |
| Freeze một phần | 81,77 ± 0,12 | **87,25 ± 0,14** |
| Full finetune | 85,65 ± 0,01 | **88,62 ± 0,10** |

(top-1 trên test, %, mean ± std qua 3 seed)

Câu hỏi 1 có câu trả lời rõ. Train từ đầu, ResNet-50 hơn DeiT-S 15 điểm, vì convolution đã có sẵn tính cục bộ còn ViT phải tự học điều đó từ 68 nghìn ảnh. Có pretrain ImageNet thì thứ hạng đảo lại ở cả ba chế độ: pretrain giúp DeiT-S thêm 21,5 điểm, giúp ResNet-50 chỉ 3,2 điểm.

Với câu hỏi 2, freeze một phần của DeiT-S giữ được 98,5% accuracy của full finetune với một nửa số tham số trainable và gần một nửa VRAM. Điều nhóm không lường trước là freeze một phần không nhanh hơn: mọi chế độ có backbone train được đều mất khoảng 66 giây mỗi epoch, vì thời gian bị chặn bởi khâu đọc ảnh và augmentation trên CPU chứ không phải GPU. Chỉ freeze toàn bộ, nhờ trích feature một lần, mới tiết kiệm thời gian thật sự.

Hai mô hình nhầm gần như cùng các cặp lớp (steak và filet mignon, chocolate cake và chocolate mousse, beef và tuna tartare), và hơn một nửa số ảnh cả hai cùng sai bị đoán thành cùng một lớp. Phần lớn lỗi còn lại đến từ các món giống nhau thật sự hoặc ảnh có nhiều món. Phân tích chi tiết ở `reports/A1.2_Report.tex`.

## 9. Kết thúc phiên

Drive trên Colab ghi file qua bộ đệm, nên trước khi tắt runtime cần gọi `flush_and_unmount` để chắc chắn checkpoint và hình cuối cùng đã thực sự được lưu lên Drive. Sau đó `runtime.unassign()` trả máy lại cho Colab để không tiếp tục tốn compute unit.

Mẹo khi để máy chạy qua đêm: bấm chạy cell này ngay sau khi bắt đầu cell 6. Colab chạy các cell lần lượt, nên cell này sẽ chờ đến khi train xong rồi mới tắt runtime.

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()
print('Đã flush Drive, tắt runtime.')
runtime.unassign()